In [1]:
import csv

# open and read the csv file
f = open("quiz4_routing.csv")
csvfile = csv.DictReader(f, delimiter=',')
headers = csvfile.fieldnames

table = []
for row in csvfile:
    table.append(row)
    
f.close()

# define the set of routes
Routes = headers[:]
Routes.remove('Distribution Center')
Routes.remove('Demand (lbs)')

# read the parameters
Demand = {}        # Demand[j] is the demand for distribution center j
IsInRoute = {}     # IsInRoute[i,j] = 1 if route i includes distribution center j
Length = {}        # Length[i] is the length of route i
for row in table:
    dc = row['Distribution Center']
    if dc != 'n/a':
        Demand[dc] = float(row['Demand (lbs)'])
        for r in Routes:
            if row[r] != '':
                IsInRoute[(r,dc)] = int(row[r])
    else:
        for r in Routes:
            Length[r] = float(row[r])
DCs = Demand.keys()

In [2]:
print(headers)

['Distribution Center', 'Demand (lbs)', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43']


In [3]:
from docplex.mp.model import Model
mdl = Model()

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [4]:
# variables
select = mdl.binary_var_dict(Routes, name='Route')
DCroute = mdl.binary_var_matrix(DCs, Routes, name='DC')

In [5]:
# objective
mdl.minimize(mdl.sum(Length[i]*select[i] for i in Routes))

In [6]:
CoverDCs = {}
for j in Demand:
    CoverDCs[j] = mdl.add_constraint(mdl.sum(select[i] for i in Routes if (i,j) in IsInRoute) == 1)
    

In [7]:
CoverDCs['1']

docplex.mp.LinearConstraint[](Route_1+Route_16+Route_20+Route_21+Route_23+Route_27+Route_28+Route_30+Route_33+Route_34+Route_35+Route_41,EQ,1)

In [8]:
# Constraint - Truck Capcity

TruckCapacity = 30000
CoverRoute ={}
for j in Routes:
    CoverRoute[j] = mdl.add_constraint(mdl.sum(DCroute[i,j]*Demand[i] for i in Demand if (j,i) in IsInRoute) <= TruckCapacity)
    

In [9]:
CoverRoute['43']

docplex.mp.LinearConstraint[](15000DC_7_43+10000DC_10_43,LE,30000)

In [10]:
# solve
mdl.solve()
mdl.get_solve_details()

docplex.mp.SolveDetails(time=0.0276289,status='integer optimal solution')

In [11]:
mdl.print_solution()

objective: 7922.000
status: OPTIMAL_SOLUTION(2)
  Route_5=1
  Route_6=1
  Route_9=1
  Route_17=1
  Route_24=1
  Route_26=1
  Route_27=1


In [12]:
# inspect how often each DCs is covered
for j in DCs:
    print("DCs " + j + ": " + str(CoverDCs[j].lhs.solution_value))

DCs 1: 1.0
DCs 2: 1.0
DCs 3: 1.0
DCs 4: 1.0
DCs 5: 1.0
DCs 6: 1.0
DCs 7: 1.0
DCs 8: 1.0
DCs 9: 1.0
DCs 10: 1.0
DCs 11: 1.0
DCs 12: 1.0
DCs 13: 1.0
DCs 14: 1.0
DCs 15: 1.0
